# 概念、文档、图片：分类型 Dataset 与显式关联

**业务主线：概念 → 关联原始文档／图片 → 共享处理 → 提炼知识。** 每种来源文件有自己的输入字段；同schema分片适配为同类业务Dataset。表中直接展示业务字段，不再使用kind＋record通用包裹。

```text
concept_sources → concepts → selected_concepts
                               ├─ JOIN document_links → selected_document_links → documents → document_texts → clean_documents
                               └─ JOIN image_links    → selected_image_links    → images    → image_checks
                               ↓ 分别计数／按需汇集
                         concept_coverage → 知识算子
                               ↓
           knowledge + concept_knowledge + knowledge_sources + image_support + knowledge_tasks
```

文档、图片分别关联，避免一个概念的10篇文档×20张图片展开成200行。共享资料使用EXISTS去重选入，只读取／清洗一次。没有资料的概念保留，未匹配的文档和图片分别保存。

当前demiflow提供惰性Dataset和map_async，没有通用join API：本流程用下面可直接查看的SQL关联，结果游标作为demiflow Dataset输入。SQL及实体表实际被算子消费，不是展示用的影子表。

小批和全量同一套算子：IDS=None选全部已读概念；SAMPLE_RATE=1不采样；MAX_RECORDS_PER_SOURCE=None读完整文件；SOURCE_PATHS=None选择全部支持来源。仍不自动全量运行或出题。

**ID含义**：concept_ref是尚未跨源合并的概念引用；doc_id与image_id是各自来源版本ID；knowledge_id是窗口任务ID＋局部事实ID，避免跨窗口重名。来源文件中的稳定标识与本次版本标识不混淆。图片SHA相同可有多个来源记录；当前保证同一来源记录不因多概念而重复处理，尚未实现跨来源所有相同字节的共享检查。


## 配置与显示

默认view_saved仅读取dataset_pilot_v1。MODE=run时每个执行cell真正调用demiflow；配置或处理代码变化需新RUN_NAME。显示limit或抽样只控制展示。RUN_MODEL_CELLS=False不调用模型。


In [ ]:
from pathlib import Path
import sys, importlib, json, sqlite3, random
PROJECT=Path('/yzp/zhaozy/yangzepeng/0905/demiwtg')
if str(PROJECT) not in sys.path:sys.path.insert(0,str(PROJECT))
from curation.v4.dataset_flow import DatasetFlow, JOINS
from curation.v4.datasets import SCHEMAS, SOURCE_SCHEMAS
from curation.v4.notebook_debug import show, detail
from curation.v4.pipeline import DEFAULT
MODE='view_saved'
RUN=PROJECT/'state/curation/v4/dataset_pilot_v1'
DATASET=PROJECT/'datasets/demiwtg'
IDS=['legacy:木兰','legacy:芦笙','qid:Q1']
SAMPLE_RATE=1.0
MAX_RECORDS_PER_SOURCE=50_000
SOURCE_PATHS=[DATASET/x for x in ['meta/concepts.json','meta/qid_concepts.fat.jsonl.gz','meta/docs.jsonl','meta/images.jsonl','corpus/pages-en-part1.jsonl.gz']]
# 新运行默认直接从datasets读取。可选RAW_RUN只复用本流程冻结的原始解析，不复用旧clean_docs或清洗结果。
RAW_RUN=None
RUN_MODEL_CELLS=False
MODEL_CONFIG={**DEFAULT,'max_calls':12}
flow=(DatasetFlow.open_saved(RUN) if MODE=='view_saved' else DatasetFlow(RUN,DATASET,SOURCE_PATHS,IDS,SAMPLE_RATE,max_records_per_source=MAX_RECORDS_PER_SOURCE,raw_run=RAW_RUN))
async def step(name):
    if MODE=='run':print(await flow.step(name))
    else:print(name,'只读保存结果')
def schema(*tables):
    for table in dict.fromkeys(tables):
        print(table);show(flow.describe(table),limit=100)
def preview(table,limit=100,sample=False,seed=42):
    if not sample:return flow.view(table,limit)
    if limit<0:raise ValueError('limit must be nonnegative')
    # 仅抽rowid，最后读取入选行；不为查看而反序列化全库正文。
    ids=[];rng=random.Random(seed)
    for i,(key,) in enumerate(flow.tables.db.execute(f'SELECT rowid FROM "{table}"')):
        if len(ids)<limit:ids.append(key)
        else:
            j=rng.randrange(i+1)
            if j<limit:ids[j]=key
    result=[]
    for key in ids:result.extend(flow.tables.rows(table,f'SELECT * FROM "{table}" WHERE rowid=?',(key,)))
    return result
print('实际配置');detail(flow.config)
print('原始解析来源',flow.raw_run)


## 0．按来源schema读取文件，形成各自的Dataset · AdaptSource

**输入**：下面列出的datasets来源文件及原始字段。同类Wiki分片可共同进入documents。内部可用已冻结原始解析缓存；业务适配器接收来源自身字段及_source_id/_provenance定位，没有record嵌套。

**处理**：每种schema独立适配，保存概念信息、文档、图片及原有关系。Wiki页有QID时直接关联；没有QID时按(lang,page_id)连接concept_pages，多个概念匹配保留歧义。原始额外字段保存在内部source_originals用于追溯，不强行塞进业务表。

**输出**：concept_sources、concepts、concept_pages、documents、images、document_links、image_links，以及未匹配表。image_roles、gallery_captions、concept_edges独立保留；当前尚未把这些辅助关系全部接入知识提示词。


In [ ]:
show([{'path':s['path'],'source_schema':s['kind'],**SOURCE_SCHEMAS[s['kind']]} for s in flow.raw.sources],limit=100)
await step('read_sources')
concepts=flow.dataset_for('concepts')
documents=flow.dataset_for('documents')
images=flow.dataset_for('images')
print(type(concepts))

In [ ]:
schema('concept_sources','concepts','concept_pages','documents','images','document_links','image_links')
show(preview('concept_sources',sample=True))
show(preview('documents',sample=True),columns=['doc_id','title','url','path','format','lang','page_id','qid','source_path','source_row'])
show(preview('images',sample=True))
with sqlite3.connect(f'file:{flow.raw_run}/records.sqlite?mode=ro',uri=True) as db:
    show([json.loads(r[0]) for r in db.execute('SELECT body FROM source_status')])

### 来源输入字段说明

| 来源schema | 原始输入字段及含义 |
| --- | --- |
| legacy_concepts | name：现有概念名主键；aliases：别名；carriers：采集载体；taxonomy：分类挂载快照 |
| qid_concepts / qid_concepts_base | qid：外部概念标识；en/zh：语言页面信息，其中page_id是站点页面ID，title是标题；其他原字段完整留存 |
| legacy_docs | path：保存正文的路径；title：标题；url：来源页；concepts/instances：采集端关联概念名 |
| wiki_pages | lang：站点语言；page_id：页面ID；qid：可空外部概念ID；title：标题；sections：保存章节，含title/text等原始字段 |
| legacy_images / qid_images | path/blob_path：本地字节路径；sha256：来源标记哈希；caption：已有描述；url/content_url：页面／图片地址；concepts/instances或qid：来源概念关联 |
| image_roles | qid：概念；property：来源属性；role：图片角色；commons_file：Commons文件名 |
| gallery_captions | qid：概念；commons_file：图片文件名；caption：原图库图注 |
| qid_graph | from/to：起止QID，其余原始关系字段保存在concept_edges.relation |

适配器输入追加_source_id和_provenance仅用于来源定位，不用它们取代实际业务字段。原始记录中额外字段仍保存在source_originals；可以按各业务表的source_id回查，不会被清洗或投影删除。


## 1．过滤／采样概念 · SelectConcept

**输入**：concepts、IDS、SAMPLE_RATE和固定种子。概念来源记录可多条，concepts按concept_ref去重；不同来源身份不按名称自动合并。

**输出**：selected_concepts保存全部选择决定；missing_concepts保存未在已读范围找到的指定ID。采样单位是概念，不是文档行。


In [ ]:
schema('concepts','selected_concepts','missing_concepts')
await step('select_concepts')

In [ ]:
show(preview('selected_concepts',sample=True))
show(list(flow.tables.rows('selected_concepts','SELECT * FROM selected_concepts WHERE selected=1 LIMIT 100')))
show(preview('missing_concepts'))

## 2．关联入选概念与文档

**输入**：selected_concepts及document_links。

**关联键**：concept_ref；只有selected=1且来源映射无歧义的关系通过。保持一行一个概念—文档关系，不与另一种材料展开组合。

**输出**：selected_document_links。unmatched_documents保存原始资料中无关联、未知概念或歧义对应，与“未入选本轮”区分。读取资料再按doc_id连接documents，使用EXISTS，多个概念共享资料不会产生重复读取行。


In [ ]:
schema('selected_concepts','document_links','selected_document_links','unmatched_documents')
print(JOINS['selected_document_links'])
await step('join_documents')

In [ ]:
show(preview('selected_document_links',sample=True))
show(preview('unmatched_documents',sample=True))

## 3．关联入选概念与图片

**输入**：selected_concepts及image_links。

**关联键**：concept_ref；只有selected=1且来源映射无歧义的关系通过。保持一行一个概念—图片关系，不与另一种材料展开组合。

**输出**：selected_image_links。unmatched_images保存原始资料中无关联、未知概念或歧义对应，与“未入选本轮”区分。读取资料再按image_id连接images，使用EXISTS，多个概念共享资料不会产生重复读取行。


In [ ]:
schema('selected_concepts','image_links','selected_image_links','unmatched_images')
print(JOINS['selected_image_links'])
await step('join_images')

In [ ]:
show(preview('selected_image_links',sample=True))
show(preview('unmatched_images',sample=True))

## 4．读取文档全文 · ReadDocument

**输入**：documents与selected_document_links按doc_id进行EXISTS关联；每篇入选文档只出现一次。

**处理**：普通文档读取path指向的原始字节；Wiki按保存的sections串联全文。文件缺失或编码错误保留read_error。

**输出**：document_texts。此算子没有图片分支，也不接收概念ID列表。


In [ ]:
schema('documents','selected_document_links','document_texts')
print(JOINS['documents_to_read'])
await step('read_documents')

In [ ]:
show(preview('document_texts',sample=True),limit=100,chars=200)

## 5．清除明确页面外壳，保留原文位置 · CleanDocument

**输入**：document_texts中的doc_id、全文和读取状态。

**输出**：clean_documents中的清洗全文、内容块、定位、保留／排除理由、质量提示。通过doc_id回连documents获得来源，不携带一个含所有类型的record。

保守清洗仍可能残留导航／模板；cleaned_candidate不代表来源可靠。blocks内raw_start/raw_end/raw_text为输入全文位置和原文，clean_start/clean_end为清洗正文位置，decision/reason为处理决定，links/images保存来源线索。字符偏移对应整块，不冒充逐字符映射。


In [ ]:
schema('document_texts','clean_documents')
await step('clean_documents')

In [ ]:
clean_sample=preview('clean_documents',sample=True)
show(clean_sample,columns=['doc_id','status','warnings','counts','version'])
if clean_sample:
    chosen=clean_sample[0]
    original=next(flow.tables.rows('document_texts','SELECT * FROM document_texts WHERE doc_id=?',(chosen['doc_id'],)))
    detail({'doc_id':chosen['doc_id'],'original':original['text'],'cleaned':chosen['text']})
    show(chosen['blocks'],limit=100)

## 6．核对图片字节 · CheckImage

**输入**：images与selected_image_links按image_id进行EXISTS关联。

**输出**：image_checks。核对路径、实际哈希与解码，缺图或失败保存状态。这里没有图像模型调用，不能据此判定图片支持知识。


In [ ]:
schema('images','selected_image_links','image_checks')
print(JOINS['images_to_check'])
await step('check_images')

In [ ]:
show(preview('image_checks',sample=True))

## 7．按概念分别统计文档、图片及缺口

**输入**：selected_concepts作为保留全部入选概念的主表，分别关联文档／图片关系和处理结果。

**输出**：concept_coverage。不先连接文档×图片再计数，避免重复统计；没有资料的概念仍输出0和no_materials_in_read_scope。数量只适用于source_status记录的范围；存在资料不等于存在可靠知识。


In [ ]:
schema('selected_concepts','selected_document_links','selected_image_links','document_texts','image_checks','concept_coverage')
print(JOINS['concept_coverage'])
await step('summarize_concepts')

In [ ]:
show(preview('concept_coverage'))

## 按概念展开资料（只改变查看范围）

分别展示文档和图片，limit只控制展示。修改CONCEPT_REF不改变运行采样或触发模型。


In [ ]:
CONCEPT_REF='legacy:芦笙'
show(list(flow.tables.rows('documents','SELECT d.* FROM documents d JOIN selected_document_links l USING(doc_id) WHERE l.concept_ref=? LIMIT 100',(CONCEPT_REF,))))
show(list(flow.tables.rows('images','SELECT i.* FROM images i JOIN selected_image_links l USING(image_id) WHERE l.concept_ref=? LIMIT 100',(CONCEPT_REF,))))

## 8．判定概念歧义与资料是否相关

输入沿用前一步模型处理结果；identity从入选概念与独立资料表按需汇集有界窗口。旧模型算子的输入兼容包仅存在于边界，提示词与调用缓存继续复用。

输出在旧阶段文件中保存完整请求依赖；export另写knowledge、concept_knowledge、knowledge_sources、image_support、knowledge_tasks。无资料概念产生blocked任务，不发空请求。跨窗口联合整合仍未实现，分表或汇总不代表知识核验通过。

下面是保留的具体模型步骤契约。material_pack等名称仅属于内部模型兼容接口，业务资料仍以上面的分类型表为准。


输入：按已有概念汇集的一窗口原文与清洗材料，加相应概念信息。模型只收到最多12篇500字符清洗预览和最多2条图片元数据，不看像素。输出：identity状态、target_label、接受／拒绝及理由、实际预览材料和未查ID。当前关联筛选仍有二分状态和预览不足的限制。

| 输入／输出字段 | 含义 |
| --- | --- |
| `record_id / case_id` | 当前材料窗口的标识，绑定概念及记录ID列表。 |
| `bundle.request` | 窗口对应的概念名或QID；不是入口查询条件。 |
| `blocked` | 前一步阻塞信息；中间阶段跳过，export仍记录。 |
| `result` | 下方投影展示该步骤的结果对象；完整输出保存在同名SQLite阶段及stages目录。 |


本批尚未调用模型。默认只查看已有输出；真正执行需要MODE=run且RUN_MODEL_CELLS=True。

In [ ]:
if MODE=='run' and RUN_MODEL_CELLS:
    print(await flow.knowledge_step('identity',MODEL_CONFIG))
else:
    print('identity: 未执行模型，只读历史输出')

## 9．去重、按预算选文与截取正文

输入沿用前一步模型处理结果；identity从入选概念与独立资料表按需汇集有界窗口。旧模型算子的输入兼容包仅存在于边界，提示词与调用缓存继续复用。

输出在旧阶段文件中保存完整请求依赖；export另写knowledge、concept_knowledge、knowledge_sources、image_support、knowledge_tasks。无资料概念产生blocked任务，不发空请求。跨窗口联合整合仍未实现，分表或汇总不代表知识核验通过。

下面是保留的具体模型步骤契约。material_pack等名称仅属于内部模型兼容接口，业务资料仍以上面的分类型表为准。


输入：身份已接受的材料。程序去重正文／图片、按页面与材料ID顺序选最多2篇，每篇最多6,500字符、合计13,000字符，并准备图片。输出material_pack：passages片段及引文位置、images可用图片、duplicates重复关系、omissions未读范围、image_gaps缺图。不是按来源权威性选文，后续仍需改进章节覆盖。

| 输入／输出字段 | 含义 |
| --- | --- |
| `record_id / case_id` | 当前材料窗口的标识，绑定概念及记录ID列表。 |
| `bundle.request` | 窗口对应的概念名或QID；不是入口查询条件。 |
| `blocked` | 前一步阻塞信息；中间阶段跳过，export仍记录。 |
| `result` | 下方投影展示该步骤的结果对象；完整输出保存在同名SQLite阶段及stages目录。 |


本批尚未调用模型。默认只查看已有输出；真正执行需要MODE=run且RUN_MODEL_CELLS=True。

In [ ]:
if MODE=='run' and RUN_MODEL_CELLS:
    print(await flow.knowledge_step('organize',MODEL_CONFIG))
else:
    print('organize: 未执行模型，只读历史输出')

## 10．联合正文提取知识候选

输入沿用前一步模型处理结果；identity从入选概念与独立资料表按需汇集有界窗口。旧模型算子的输入兼容包仅存在于边界，提示词与调用缓存继续复用。

输出在旧阶段文件中保存完整请求依赖；export另写knowledge、concept_knowledge、knowledge_sources、image_support、knowledge_tasks。无资料概念产生blocked任务，不发空请求。跨窗口联合整合仍未实现，分表或汇总不代表知识核验通过。

下面是保留的具体模型步骤契约。material_pack等名称仅属于内部模型兼容接口，业务资料仍以上面的分类型表为准。


输入模型：concept及passages中的source_id、text、source_family、start、end；不传全部原文映射。输出extraction：facts（fact_id、statement、conditions、exceptions、evidence中的source_id与quote），unresolved_conflicts（source_ids、issue、needed_evidence）和coverage_note。引文匹配不等于正确。

| 输入／输出字段 | 含义 |
| --- | --- |
| `record_id / case_id` | 当前材料窗口的标识，绑定概念及记录ID列表。 |
| `bundle.request` | 窗口对应的概念名或QID；不是入口查询条件。 |
| `blocked` | 前一步阻塞信息；中间阶段跳过，export仍记录。 |
| `result` | 下方投影展示该步骤的结果对象；完整输出保存在同名SQLite阶段及stages目录。 |


本批尚未调用模型。默认只查看已有输出；真正执行需要MODE=run且RUN_MODEL_CELLS=True。

In [ ]:
if MODE=='run' and RUN_MODEL_CELLS:
    print(await flow.knowledge_step('extract',MODEL_CONFIG))
else:
    print('extract: 未执行模型，只读历史输出')

## 11．对照来源修订候选并记录冲突

输入沿用前一步模型处理结果；identity从入选概念与独立资料表按需汇集有界窗口。旧模型算子的输入兼容包仅存在于边界，提示词与调用缓存继续复用。

输出在旧阶段文件中保存完整请求依赖；export另写knowledge、concept_knowledge、knowledge_sources、image_support、knowledge_tasks。无资料概念产生blocked任务，不发空请求。跨窗口联合整合仍未实现，分表或汇总不代表知识核验通过。

下面是保留的具体模型步骤契约。material_pack等名称仅属于内部模型兼容接口，业务资料仍以上面的分类型表为准。


输入：同一批passages及上一轮extraction候选。同一本地模型复查，不自动补读全文。输出knowledge字段与extraction结构相同，增加changes（fact_id、action、reason）。它不是独立审核，也没有检查其他窗口的冲突。

| 输入／输出字段 | 含义 |
| --- | --- |
| `record_id / case_id` | 当前材料窗口的标识，绑定概念及记录ID列表。 |
| `bundle.request` | 窗口对应的概念名或QID；不是入口查询条件。 |
| `blocked` | 前一步阻塞信息；中间阶段跳过，export仍记录。 |
| `result` | 下方投影展示该步骤的结果对象；完整输出保存在同名SQLite阶段及stages目录。 |


本批尚未调用模型。默认只查看已有输出；真正执行需要MODE=run且RUN_MODEL_CELLS=True。

In [ ]:
if MODE=='run' and RUN_MODEL_CELLS:
    print(await flow.knowledge_step('consolidate',MODEL_CONFIG))
else:
    print('consolidate: 未执行模型，只读历史输出')

## 12．核验图片对具体知识的支持

输入沿用前一步模型处理结果；identity从入选概念与独立资料表按需汇集有界窗口。旧模型算子的输入兼容包仅存在于边界，提示词与调用缓存继续复用。

输出在旧阶段文件中保存完整请求依赖；export另写knowledge、concept_knowledge、knowledge_sources、image_support、knowledge_tasks。无资料概念产生blocked任务，不发空请求。跨窗口联合整合仍未实现，分表或汇总不代表知识核验通过。

下面是保留的具体模型步骤契约。material_pack等名称仅属于内部模型兼容接口，业务资料仍以上面的分类型表为准。


输入：knowledge.facts和material_pack.images的实际像素；原图核对后缩放到最长边1,024并JPEG编码。输出image_evidence：无图／无事实为not_run；实际调用后为machine_reviewed，内含images的caption和support逐图逐事实的status、region、supports、limitations。真实图像模型分支仍待验收。

| 输入／输出字段 | 含义 |
| --- | --- |
| `record_id / case_id` | 当前材料窗口的标识，绑定概念及记录ID列表。 |
| `bundle.request` | 窗口对应的概念名或QID；不是入口查询条件。 |
| `blocked` | 前一步阻塞信息；中间阶段跳过，export仍记录。 |
| `result` | 下方投影展示该步骤的结果对象；完整输出保存在同名SQLite阶段及stages目录。 |


本批尚未调用模型。默认只查看已有输出；真正执行需要MODE=run且RUN_MODEL_CELLS=True。

In [ ]:
if MODE=='run' and RUN_MODEL_CELLS:
    print(await flow.knowledge_step('evidence',MODEL_CONFIG))
else:
    print('evidence: 未执行模型，只读历史输出')

## 13．保存分表知识、引文、图片支持和任务状态

输入沿用前一步模型处理结果；identity从入选概念与独立资料表按需汇集有界窗口。旧模型算子的输入兼容包仅存在于边界，提示词与调用缓存继续复用。

输出在旧阶段文件中保存完整请求依赖；export另写knowledge、concept_knowledge、knowledge_sources、image_support、knowledge_tasks。无资料概念产生blocked任务，不发空请求。跨窗口联合整合仍未实现，分表或汇总不代表知识核验通过。

下面是保留的具体模型步骤契约。material_pack等名称仅属于内部模型兼容接口，业务资料仍以上面的分类型表为准。


输入：身份判断、knowledge.facts及unresolved_conflicts、image_evidence和blocked。输出export及candidates文件：case_id、concept_id、request、status、facts、冲突与图片支持。blocked保存失败；machine_candidates_ready_for_review仍是机器候选，不是已核验知识库。

| 输入／输出字段 | 含义 |
| --- | --- |
| `record_id / case_id` | 当前材料窗口的标识，绑定概念及记录ID列表。 |
| `bundle.request` | 窗口对应的概念名或QID；不是入口查询条件。 |
| `blocked` | 前一步阻塞信息；中间阶段跳过，export仍记录。 |
| `result` | 下方投影展示该步骤的结果对象；完整输出保存在同名SQLite阶段及stages目录。 |


本批尚未调用模型。默认只查看已有输出；真正执行需要MODE=run且RUN_MODEL_CELLS=True。

In [ ]:
if MODE=='run' and RUN_MODEL_CELLS:
    print(await flow.knowledge_step('export',MODEL_CONFIG))
else:
    print('export: 未执行模型，只读历史输出')

## 知识输出表与关联键

knowledge通过knowledge_id连接concept_knowledge、knowledge_sources、image_support；引文通过doc_id回查文档，图片支持通过image_id回查图片。knowledge_tasks保留任务阻塞、未解冲突和覆盖说明。当前真实小批未调用模型，因此这些表为空，不能冒称已经产出知识。


In [ ]:
schema('knowledge','concept_knowledge','knowledge_sources','image_support','knowledge_tasks')
for table in ['knowledge','concept_knowledge','knowledge_sources','image_support','knowledge_tasks']:
    print(table);show(preview(table))

## 辅助来源表及完整字段字典

图库图注、图片角色和概念图关系独立保存，当前不冒称已完整参与知识提取。未知来源额外字段在source_originals中保留原始内容与定位，供追溯及兼容使用；不会成为业务算子的通用输入。


In [ ]:
schema('image_roles','gallery_captions','concept_edges')
show([{'dataset':table,'rows':flow.tables.db.execute(f'SELECT count(*) FROM "{table}"').fetchone()[0]} for table in SCHEMAS])
print('模型请求数',len(list(RUN.rglob('request.json'))))